# Gemma (GGUF) — thử nghiệm & chuẩn bị dữ liệu OCR post-process

Mục tiêu:
1. **Path A (runtime):** tải file `.gguf`, chạy `llama-cpp-python` (giống service `app/services/local_gguf_llm.py`).
2. **Dữ liệu:** dùng JSONL trong `data/ocr_corrections/` (export từ AI service) để huấn luyện / distillation sau.
3. **Gemma 4:** khi Google/Hugging Face phát hành GGUF chính thức, chỉ cần đổi `GGUF_REPO` / tên file; luồng Colab giữ nguyên.

**Thứ tự fallback trên server:** Local GGUF → Gemini API → rule-based (`text_postprocessor`).

## 1. Cài đặt

- **CPU:** wheel mặc định của `llama-cpp-python`.
- **GPU Colab:** sau `pip install`, có thể cần build lại với CUDA theo [llama-cpp-python — CUDA](https://github.com/abetlen/llama-cpp-python#installation-with-cuda).

In [ ]:
%pip install -q llama-cpp-python huggingface_hub

## 2. Tải GGUF mẫu (Gemma-class)

Đổi `REPO_ID` và `FILENAME` theo repo bạn chọn (ví dụ các bản quant từ `bartowski` hoặc `unsloth` trên Hugging Face).

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

# Ví dụ — thay bằng Gemma / Gemma2 / Gemma3 GGUF bạn có quyền truy cập
REPO_ID = "google/gemma-2-2b-it-GGUF"  # hoặc repo GGUF khác
FILENAME = "gemma-2-2b-it-Q4_0.gguf"   # hoặc Q4_K_M, …

models_dir = Path("/content/models")
models_dir.mkdir(parents=True, exist_ok=True)
gguf_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, local_dir=str(models_dir))
print("GGUF:", gguf_path)

## 3. Inference thử (chat JSON — cùng ý tưởng với `SYSTEM_PROMPT` service)

`chat_format`: với Gemma 2 thường dùng `gemma`; Gemma 3 có thể khác — xem gợi ý trong card model GGUF.

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=gguf_path,
    n_ctx=8192,
    n_gpu_layers=0,  # Colab GPU: thử 35 hoặc -1 tùu VRAM
    verbose=False,
    chat_format="gemma",  # đổi thành "gemma-2" nếu model yêu cầu
)

system = """Bạn là chuyên gia OCR hậu kỳ. Trả về JSON hợp lệ, không markdown.
Schema: {"name": str, "brand": str, "ingredients": str, "category": str, ...}"""
user = """RAW OCR:\nTHIT NAC VISSAN 500G\nThanh phan: thit heo, muoi"""

out = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ],
    temperature=0.1,
    max_tokens=1024,
)
print(out["choices"][0]["message"]["content"])

## 4. Fine-tune / merge / GGUF (hướng dẫn ngắn)

1. **Dataset:** đọc `data/ocr_corrections/corrections_YYYY-MM-DD.jsonl` — mỗi dòng có `input.raw_text`, `input.regions`, `output` (JSON đích).
2. **Format huấn luyện:** chuyển thành cặp instruction/response (chat) hoặc Alpaca; nhắc model luôn trả JSON theo schema trong `USER_PROMPT_TEMPLATE` của `llm_postprocessor.py`.
3. **LoRA / full SFT:** dùng `transformers` + `peft` hoặc Unsloth trên Colab Pro GPU.
4. **Export GGUF:** sau khi có adapter/merged weights, dùng `llama.cpp` `convert_hf_to_gguf.py` (hoặc script đi kèm repo quant) để xuất `.gguf` Q4_0 / Q4_K_M.
5. **Deploy:** copy file `.gguf` lên máy chạy AI service, set `AI_LLM_GGUF_PATH` và tùy chỉnh `AI_LLM_CHAT_FORMAT`.

Lệnh convert (tham khảo — cần clone `llama.cpp` và đúng nhánh hỗ trợ kiến trúc Gemma của bạn):
```bash
python convert_hf_to_gguf.py /path/to/merged_hf_model --outfile model.gguf --outtype q4_0
```